# Commodity Futures Pairs Trading — Time Series Final Project
**Course:** Time Series Analysis | **Dataset:** Commodity Futures Daily Prices (2000–2023)


## Project Overview
This project implements a full **statistical arbitrage pairs trading strategy** on commodity futures, grounded in cointegration theory. We identify pairs sharing a long-run equilibrium, model the spread as an Ornstein-Uhlenbeck process, generate Z-score-based trading signals, and validate rigorously using walk-forward cross-validation.

| Step | Concept | Purpose |
|---|---|---|
| 1 | EDA & Data Cleaning | Understand the dataset; remove unusable columns; fill gaps |
| 2 | Stationarity (ADF + KPSS) | Confirm each price is I(1) — hard prerequisite for cointegration |
| 3 | Engle-Granger Cointegration | Screen all 190 pairs for a stationary linear combination |
| 4 | Pair Selection | Filter by economics + unit compatibility; reject spurious pairs |
| 5 | Feature Engineering | Hedge ratio β, spread, Z-score, half-life |
| 6 | Three Models | Static OLS, Rolling OLS, VECM — compare hedge ratio approaches |
| 7 | Signal Generation & Backtest | Stateful Z-score signals with **stop loss** and per-trade win rate |
| 8 | Spread Diagnostics | ADF, histogram, ACF, QQ plot |
| 9 | Walk-Forward CV | Time-aware validation — no look-ahead bias |
| 10 | OOS Backtest (2020–2023) | Stress test through COVID crash + Russia-Ukraine shock |
| 11 | Model Comparison | Ranked by OOS Sharpe, CV stability, max drawdown |
| 12 | Granger Causality | Which asset in each pair leads? Does lagged X help forecast Y? |
| 13 | OU Forecasting — Train/Test | Fit OU on 2000→Feb 2023, test on last 6 months, overlay actual |
| 14 | OU Forecast → 2025 | Monte Carlo fan chart for full future convergence prediction |


## 0. Imports

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from itertools import combinations
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss, coint, acf, grangercausalitytests
from statsmodels.tsa.vector_ar.vecm import VECM
from statsmodels.regression.rolling import RollingOLS
import statsmodels.api as sm

plt.rcParams.update({'figure.figsize':(14,5),'axes.grid':True,'grid.alpha':0.3, 'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
np.random.seed(42)

## 1. Data Loading & Exploratory Data Analysis

In [ ]:
df = pd.read_csv('commodity_futures.csv', parse_dates=['Date'], index_col='Date')
print(f'Shape: {df.shape}  |  {df.index.min().date()} → {df.index.max().date()}')
df.head()

In [ ]:
missing = (df.isnull().sum() / len(df) * 100).round(2)
print('Missing % per column (only those >0%):')
print(missing[missing>0].sort_values(ascending=False))

### Cleaning Decisions
- GASOLINE dropped: 24.5% missing — imputing this much data would inject more noise than signal.
- Forward-fill then back-fill for remaining gaps: futures exchanges have different holiday schedules. When one exchange is closed, the last traded price is the best available estimate of fair value — forward-fill captures this exactly.

In [ ]:
df_clean = df.drop(columns=['GASOLINE'], errors='ignore').ffill().bfill()
print(f'Clean shape: {df_clean.shape}  |  Remaining nulls: {df_clean.isnull().sum().sum()}')

In [ ]:
# Full price history — spot structural changes and trending periods
fig, axes = plt.subplots(6,4,figsize=(22,20)); axes=axes.flatten()
for i,col in enumerate(df_clean.columns):
    axes[i].plot(df_clean.index, df_clean[col], lw=0.8)
    axes[i].set_title(col, fontsize=9, fontweight='bold')
    axes[i].xaxis.set_major_locator(mdates.YearLocator(5))
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    axes[i].tick_params(axis='x', labelsize=7)
for j in range(i+1, len(axes)): axes[j].set_visible(False)
plt.suptitle('Commodity Futures — Daily Closing Prices (2000–2023)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# Sector-normalised indices (base=100 at start)
# Purpose: removes price-level differences so we can compare relative performance
sectors = {
    'Energy':       ['WTI CRUDE','BRENT CRUDE','NATURAL GAS','LOW SULPHUR GAS OIL','ULS DIESEL'],
    'Metals':       ['GOLD','SILVER','COPPER','ALUMINIUM','ZINC','NICKEL'],
    'Agricultural': ['SOYBEANS','CORN','WHEAT','HRW WHEAT','SOYBEAN OIL',
                     'SOYBEAN MEAL','SUGAR','COFFEE','COTTON','LEAN HOGS','LIVE CATTLE']
}
fig, axes = plt.subplots(1,3,figsize=(21,5))
for ax,(sec,cols) in zip(axes,sectors.items()):
    ca = [c for c in cols if c in df_clean.columns]
    (df_clean[ca].div(df_clean[ca].iloc[0])*100).plot(ax=ax, lw=0.9)
    ax.set_title(f'{sec} Sector', fontweight='bold')
    ax.set_ylabel('Index (Base=100)'); ax.legend(fontsize=7)
plt.suptitle('Sector-Normalised Price Indices (2000–2023)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Correlation matrix — visual pre-screen only
corr = df_clean.corr()
plt.figure(figsize=(16,13))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, annot_kws={'size':6}, vmin=-1, vmax=1)
plt.title('Pairwise Correlation Matrix\n'
          '⚠  High correlation ≠ cointegration — formal test in Section 3',
          fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

## 2. Stationarity Testing — ADF + KPSS

In [ ]:
def adf_p(s):  return adfuller(s.dropna(), autolag='AIC')[1]
def kpss_p(s): return kpss(s.dropna(), regression='c', nlags='auto')[1]

rows = []
for col in df_clean.columns:
    s = df_clean[col].dropna()
    al = adf_p(s)
    ad = adf_p(s.diff().dropna())
    kl = kpss_p(s)
    rows.append({'Commodity': col,
                 'ADF p (level)': round(al,4),
                 'ADF p (Δ)': round(ad,4),
                 'KPSS p (level)': round(kl,4),
                 'I(1)?': (al > 0.05) and (ad < 0.05)})
stat_df = pd.DataFrame(rows)
print(stat_df.to_string(index=False))

In [ ]:
i1_cols = stat_df[stat_df['I(1)']==True]['Commodity'].tolist() if 'I(1)' in stat_df else \
          stat_df[stat_df['I(1)?']==True]['Commodity'].tolist()
excl    = [c for c in df_clean.columns if c not in i1_cols]
print(f'I(1) — proceed to cointegration ({len(i1_cols)}): {i1_cols}')
print(f'\nExcluded ({len(excl)}): {excl}')
print('\nNATURAL GAS: stationary at levels — strong seasonal/storage cycles create mean reversion.')
print('LEAN HOGS: stationary — livestock herd-cycle biology constrains prices to mean-revert.')
df_coint = df_clean[i1_cols].copy()

## 3. Cointegration Screening — All Pairs

### Engle-Granger Two-Step Test
1. Regress Y on X via OLS to estimate hedge ratio β. 

2. Run ADF on residuals. If residuals are stationary (p < 0.05), pair is cointegrated, the spread Y − βX is mean-reverting.

### Why Cointegration and Not Just Correlation
Two I(1) series (random walks) can show 0.90+ correlation purely because both trend upward over time — spurious correlation. Cointegration explicitly tests whether a linear combination is stationary, meaning the deviation from the relationship is bounded. That bounded deviation is what we trade.

### Spurious Cointegrations
Many pairs pass the statistical test but have no economic link (e.g. WTI / HRW Wheat, p = 0.0007). They appeared cointegrated because both rose with the commodity supercycle 2000–2022. We filter these out later on using economic rationale.

In [ ]:
pair_res = []
cols = df_coint.columns
all_pairs = list(combinations(cols, 2))
print(f'Testing {len(all_pairs)} pairs...')
for c1,c2 in all_pairs:
    s1 = df_coint[c1].dropna(); s2 = df_coint[c2].dropna()
    idx = s1.index.intersection(s2.index)
    if len(idx) < 252: continue
    try:
        sc,pv,_ = coint(s1.loc[idx], s2.loc[idx])
        pair_res.append({'Asset 1':c1,'Asset 2':c2,'Score':round(sc,4),'P-Value':round(pv,4)})
    except: pass
pairs_df = pd.DataFrame(pair_res).sort_values('P-Value').reset_index(drop=True)
sig = pairs_df[pairs_df['P-Value'] < 0.05]
print(f'Cointegrated (p<0.05): {len(sig)} of {len(pairs_df)}')
print('\nTop 20 by cointegration strength:')
print(sig.head(20).to_string(index=False))

In [ ]:
# p-value heatmap across all pairs — green = strong cointegration
pval_mat = pd.DataFrame(index=cols, columns=cols, dtype=float)
for _,r in pairs_df.iterrows():
    pval_mat.loc[r['Asset 1'],r['Asset 2']] = r['P-Value']
    pval_mat.loc[r['Asset 2'],r['Asset 1']] = r['P-Value']
pval_mat = pval_mat.fillna(1.0); np.fill_diagonal(pval_mat.values, 0.0)
plt.figure(figsize=(15,12))
mask = np.triu(np.ones_like(pval_mat, dtype=bool))
sns.heatmap(pval_mat.astype(float), mask=mask, cmap='RdYlGn_r',
            vmin=0, vmax=0.1, annot=True, fmt='.3f',
            annot_kws={'size':6}, linewidths=0.3)
plt.title('Engle-Granger p-values (darker green = stronger cointegration | cut-off 0.05)',
          fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Pair Selection — Economics + Unit Compatibility

### Three-Filter Process (in order)
1. Statistical: p < 0.05 (Engle-Granger confirms cointegration)
2. Economic: a genuine supply-chain or substitution mechanism that forces mean reversion — rules out pairs that are statistically cointegrated by coincidence
3. Unit compatibility: prices must be in comparable units so that β and the spread carry economic meaning, not just unit conversion arithmetic


### Selected Pairs
| Pair | p-value | Why Valid |
|---|---|---|
| **WTI / Brent Crude** | ~0.001 | Same crude oil benchmark, different delivery points. Physical arbitrage via tankers forces convergence. β ≈ 1.11 (Brent at 11% premium) |
| **Low Sulphur Gas Oil / ULS Diesel** | ~0.000 | Chemically almost identical refined products. Refinery switching enforces parity. Strongest pair in dataset. |
| **Soybeans / Soybean Meal** | ~0.016 | Meal is extracted from soybeans in the crush process (~44 lbs meal per bushel). Crush economics create a direct cost equilibrium. Spread = crush margin signal. |
| **Corn / Wheat** | ~0.001 | Both ¢/bushel. Feed-grain substitutes — livestock feeders switch between them based on price. β ≈ 0.76 is interpretable. |


### Rejected Pairs
| Pair | p-value | Reason Rejected |
|---|---|---|
| Gold / Silver | 0.677 | Not cointegrated — gold/silver ratio ranged 65–127× over the sample |
| Sugar / Coffee | 0.0002 | Unit-incompatible — β = 7.1 is a price-ratio artifact, spread = −895 is meaningless |
| WTI / HRW Wheat | 0.0007 | Spurious — no supply chain link; common macro trend only |
| Gas Oil / HRW Wheat | 0.0002 | Spurious — energy vs grain, completely separate markets |
| WTI / Gas Oil | — | Crude input vs refined output — structurally biased spread, not symmetric |

In [ ]:
PAIRS = [
    ('WTI CRUDE',           'BRENT CRUDE',   'Energy — Crude Oil'),
    ('LOW SULPHUR GAS OIL', 'ULS DIESEL',    'Energy — Refined Products'),
    ('SOYBEANS',            'SOYBEAN MEAL',  'Agricultural — Crush Spread'),
    ('CORN',                'WHEAT',         'Agricultural — Feed Grains'),
]

print('=== Selected Pairs — Cointegration Confirmation ===')
for a1,a2,label in PAIRS:
    row = pairs_df[pairs_df.apply(
        lambda r: set([r['Asset 1'],r['Asset 2']]) == {a1,a2}, axis=1)]
    pv = row.iloc[0]['P-Value'] if len(row) else float('nan')
    print(f'  {label} ({a1} / {a2}):  p = {pv:.4f}  {"✓" if pv<0.05 else "✗"}')

## 5. Feature Engineering

### What We Compute and Why
| Feature | Formula | Why |
|---|---|---|
| **Static β** | Full-sample OLS of Y on X | Baseline hedge ratio — minimises full-history spread variance |
| **Rolling β** (2×HL window) | OLS on trailing 2×half-life days | Adapts to shifting relationships; window sized to the pair's own reversion speed |
| **Spread** | Y − β·X | The cointegrating residual — stationary by construction |
| **Z-score** | (spread − μ) / σ over 2×HL window | Normalises spread into std dev units → universal ±2 signal |
| **Half-life** | ln(2) / κ from AR(1) on spread | How many days for spread to revert halfway back to mean |

### Half-Life Interpretation
The half-life is estimated by regressing ΔSpread_t on Spread_(t-1). The slope κ is the mean-reversion speed. Half-life = ln(2)/κ. This tells you how long a typical trade will last, which determines whether daily-frequency trading is feasible:
- **< 5 days:** too fast — transaction costs eat all profit
- **5–100 days:** ideal for daily-frequency pairs trading
- **> 100 days:** capital tied up too long; regime-change risk is high

### Using 2× Half-Life as the Rolling Window?
Using a fixed 60-day window treats all pairs the same regardless of how quickly they mean-revert. This is theoretically inconsistent:
- **Gas Oil/Diesel (HL = 5.8 days):** A 60-day window uses ~10× more history than one mean-reversion cycle. It over-smooths and responds too slowly to genuine shifts.
- **Corn/Wheat (HL = 83 days):** A 60-day window is *shorter* than one full reversion cycle — the rolling β and Z-score are estimated on a window that has not even seen one complete trade play out. This underestimates volatility.

The **2× half-life rule** ties the estimation window to the pair's own dynamics: long enough to capture ~2 full reversion cycles for stable parameter estimation, short enough to adapt to genuine regime shifts. A minimum of 20 days is enforced for numerical stability.

| Pair | Half-life | Rolling Window (2×HL, min 20) |
|---|---|---|
| Energy — Crude Oil (WTI/Brent) | 21.9 days | **44 days** |
| Energy — Refined Products (Gas Oil/Diesel) | 5.8 days | **20 days** (min enforced) |
| Agricultural — Crush Spread (Soybeans/Meal) | 85.8 days | **172 days** |
| Agricultural — Feed Grains (Corn/Wheat) | 83.0 days | **166 days** |


In [ ]:
def compute_half_life(spread):
    """AR(1) regression on spread changes to estimate mean-reversion speed κ."""
    s = spread.dropna()
    lag = s.shift(1).dropna(); diff = s.diff().dropna()
    idx = lag.index.intersection(diff.index)
    X = sm.add_constant(lag.loc[idx])
    res = sm.OLS(diff.loc[idx], X).fit()
    kappa = res.params.iloc[1]
    return round(-np.log(2)/kappa, 1) if kappa < 0 else np.nan

def hl_to_window(hl, minimum=20):
    """
    Convert half-life to rolling window size.
    Rule: window = max(round(2 * half_life), minimum)
    Rationale: 2x HL covers ~2 full reversion cycles — enough data for stable
    parameter estimation while remaining responsive to genuine regime shifts.
    A minimum of 20 days is enforced for numerical stability.
    """
    if hl is None or np.isnan(hl): return minimum
    return max(round(2 * hl), minimum)

def build_pair_features(data, a1, a2):
    """
    Compute spread, Z-score, and half-life for one pair.
    Two-pass design:
      Pass 1 — fit static OLS and compute half-life from the static spread.
      Pass 2 — use 2×HL as the rolling window for rolling OLS and Z-scores.
    This ensures the rolling window is always calibrated to each pair's own
    mean-reversion dynamics rather than a one-size-fits-all constant.
    """
    p = data[[a1,a2]].dropna().copy()
    # ── Pass 1: static OLS + half-life ──────────────────────────────────────
    X = sm.add_constant(p[a1])
    ols = sm.OLS(p[a2], X).fit()
    beta_s = ols.params[a1]
    p['spread_static'] = p[a2] - beta_s * p[a1]
    hl   = compute_half_life(p['spread_static'])   # computed BEFORE rolling window
    roll = hl_to_window(hl)                         # window derived FROM half-life
    # ── Pass 2: rolling OLS + Z-scores using pair-specific window ──────────
    rr = RollingOLS(p[a2], sm.add_constant(p[a1]), window=roll).fit()
    p['beta_rolling']   = rr.params[a1]
    p['spread_rolling'] = p[a2] - p['beta_rolling'] * p[a1]
    for sc, zc in [('spread_static','z_static'), ('spread_rolling','z_rolling')]:
        mu = p[sc].rolling(roll).mean()
        sd = p[sc].rolling(roll).std()
        p[zc] = (p[sc] - mu) / sd
    p.dropna(inplace=True)
    return p, round(beta_s, 4), hl, roll

pair_data = {}
print(f"{'Pair':<35} {'β (static)':>12} {'Half-life (days)':>18} {'Roll Window':>12}")
print('-'*80)
for a1,a2,label in PAIRS:
    p, beta, hl, roll = build_pair_features(df_clean, a1, a2)
    pair_data[(a1,a2)] = {'df':p, 'label':label, 'beta':beta, 'half_life':hl, 'roll':roll}
    print(f'{label:<35} {beta:>12} {str(hl)+" days":>18} {str(roll)+" days":>12}')

In [ ]:
# Visualise price levels, spread, and Z-score for each pair
for (a1,a2),info in pair_data.items():
    p = info['df']; label = info['label']
    fig, axes = plt.subplots(3,1,figsize=(16,11), sharex=True)
    fig.suptitle(f'{label}: {a1} vs {a2}', fontsize=13, fontweight='bold')
    # Panel 1: prices
    ax1 = axes[0]; ax1r = ax1.twinx()
    p[a1].plot(ax=ax1,  color='steelblue', lw=0.8, label=a1)
    p[a2].plot(ax=ax1r, color='tomato',    lw=0.8, label=a2)
    ax1.set_ylabel(a1, color='steelblue'); ax1r.set_ylabel(a2, color='tomato')
    lines = ax1.get_lines() + ax1r.get_lines()
    ax1.legend(lines, [l.get_label() for l in lines], loc='upper left', fontsize=9)
    ax1.set_title('Price levels — both trending together is expected and correct for I(1) series')
    # Panel 2: spread
    sp = p['spread_static']; mu = sp.mean()
    axes[1].plot(p.index, sp, color='purple', lw=0.8, label='Spread (Y − βX)')
    axes[1].axhline(mu, color='black', ls='--', lw=1, label=f'Mean = {mu:.2f}')
    axes[1].fill_between(p.index, sp, mu, where=sp>mu, alpha=0.15, color='red')
    axes[1].fill_between(p.index, sp, mu, where=sp<mu, alpha=0.15, color='green')
    axes[1].set_title(f'Spread — static β={info["beta"]}  Half-life={info["half_life"]} days')
    axes[1].legend(fontsize=9)
    # Panel 3: Z-score
    axes[2].plot(p.index, p['z_static'], color='darkorange', lw=0.7)
    for th,c in [(2,'red'),(-2,'green'),(0.5,'gray'),(-0.5,'gray')]:
        axes[2].axhline(th, color=c, ls='--', lw=0.8)
    axes[2].set_title('Z-Score (Entry ±2σ | Exit ±0.5σ)')
    plt.tight_layout(); plt.show()

### 5.1 Hedge Ratio Interpretation

- **WTI/Brent β ≈ 1.11:** Brent trades at ~11% premium to WTI. The spread Brent − 1.11×WTI oscillates around zero.
- **Gas Oil/Diesel β ≈ 0.32:** Gas Oil in $/MT, Diesel in ¢/gal. Different units, but OLS absorbs the conversion naturally. The spread is confirmed stationary by ADF.
- **Soybeans/Meal β ≈ 0.27:** Statistical hedge ratio in price-space. Spread captures the crush margin.
- **Corn/Wheat β ≈ 1.017:** Both in ¢/bushel. β=1.017 is the **statistical hedge ratio** — it measures how much Wheat moves per unit of Corn (Cov/Var), not the raw price ratio. The intercept α ≈ 115 absorbs the structural level gap between the two prices. The spread WHEAT − 1.017×CORN is stationary and mean-reverting (ADF confirmed).

> **Why β ≠ price ratio:** OLS minimises spread *variance*, not price ratio. β = Cov(WHEAT,CORN)/Var(CORN). This captures price co-movement. The intercept handles level differences. The spread mean-reverts correctly regardless.

In [ ]:
# Rolling β stability chart — how much does the hedge ratio drift?
fig, axes = plt.subplots(2,2,figsize=(18,10)); axes=axes.flatten()
for ax,((a1,a2),info) in zip(axes,pair_data.items()):
    d = info['df']
    roll = info['roll']
    d['beta_rolling'].plot(ax=ax, color='purple', lw=0.8, label=f'Rolling β ({roll}-day)')
    ax.axhline(info['beta'], color='gray', ls='--', lw=1.2,
               label=f'Static β = {info["beta"]}')
    ax.set_title(info['label'], fontweight='bold')
    ax.set_ylabel('Hedge Ratio β'); ax.legend(fontsize=9)
windows = ', '.join(f"{info['label'].split('—')[-1].strip()}: {info['roll']}d"
                    for info in pair_data.values())
plt.suptitle(f'Rolling OLS vs Static Hedge Ratio — Half-Life-Based Windows\n({windows})',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

## 6. Signal Generation & Backtesting

### Trading Rules
```
Z > +2.0  →  SHORT spread (sell Y, buy X)    ← Y unusually expensive vs X
Z < −2.0  →  LONG  spread (buy Y, sell X)    ← Y unusually cheap vs X
|Z| < 0.5 →  CLOSE position                  ← spread has mean-reverted
|Z| > 4.0 →  STOP LOSS — close immediately   ← relationship may have broken down
```

### Stop Loss — Why |Z| = 4.0
A Z-score of ±4 means the spread is 4 standard deviations from its mean — this happens in less than 0.006% of observations under a normal distribution. When it does happen, it usually signals a structural break (e.g. WTI going negative in April 2020) rather than an extreme but temporary divergence. Waiting for reversion in these cases can cause catastrophic losses. Closing at ±4 caps tail risk.

### Signal Generator is Stateful (Not Vectorised)
A naive vectorised approach (`df.loc[z > 2, 'signal'] = -1`) fires a new signal every single day the Z-score is above 2 — even if you are already in the position. This would attempt to enter the same trade 30 times in a row. The correct implementation is a **state machine** that tracks whether you are currently flat, long, or short, and only permits valid state transitions.

### P&L Normalisation
Gas Oil spreads move in hundreds of $/MT; Corn/Wheat spreads in tens of ¢/bushel. Comparing raw P&L across pairs is meaningless. Dividing by the training-period spread standard deviation converts all pairs to standardised units, making Sharpe ratios directly comparable.

### Per-Trade Win Rate
A mean-reverting strategy will always show ~50% winning *days* because the spread walks up and down randomly day-to-day. The meaningful metric is the fraction of complete **round-trips** (entry → exit) that close profitably. This should exceed 50% if the strategy has genuine edge.


In [ ]:
def generate_signals(zscore, entry=2.0, exit_=0.5, stop=4.0):
    """
    Stateful Z-score signal generator with stop loss.
    State transitions:
      flat  → long  : z < -entry
      flat  → short : z > +entry
      long  → flat  : z > -exit_  (reversion)
      short → flat  : z < +exit_  (reversion)
      any   → flat  : |z| > stop  (stop loss)
    """
    pos = 0; out = np.zeros(len(zscore))
    for i, z in enumerate(zscore.values):
        if np.isnan(z): out[i] = 0; continue
        if abs(z) > stop:         pos = 0   # stop loss — close immediately
        elif pos == 0:
            if   z < -entry:      pos =  1  # enter long
            elif z >  entry:      pos = -1  # enter short
        elif pos ==  1 and z > -exit_: pos = 0  # close long
        elif pos == -1 and z <  exit_: pos = 0  # close short
        out[i] = pos
    return pd.Series(out, index=zscore.index)

def backtest(spread, position, spread_std=None, tc_frac=0.001):
    """
    Normalised backtest with per-TRADE win rate.
    P&L is divided by spread_std so all pairs are in comparable units.
    Win rate tracks complete round-trips, not individual days.
    """
    if spread_std is None: spread_std = spread.std()
    if spread_std <= 0 or np.isnan(spread_std): spread_std = 1.0
    spread_ret = spread.diff()
    pos_lag    = position.shift(1).fillna(0)
    # Transaction cost on position changes
    tc = tc_frac * abs(position.diff().fillna(0))
    pnl = (pos_lag * spread_ret - tc) / spread_std
    pnl.fillna(0, inplace=True)
    # Per-TRADE win rate
    trade_pnls = []
    in_trade = False; trade_pnl = 0.0
    for i in range(1, len(pnl)):
        if pos_lag.iloc[i] != 0:
            trade_pnl += pnl.iloc[i]; in_trade = True
        elif in_trade:
            trade_pnls.append(trade_pnl)
            trade_pnl = 0.0; in_trade = False
    n_trades = len(trade_pnls)
    win_rate = round((np.array(trade_pnls) > 0).mean() * 100, 1) if n_trades > 0 else 0.0
    # Cumulative P&L and risk metrics
    cumulative = pnl.cumsum()
    roll_max   = cumulative.cummax()
    drawdown   = cumulative - roll_max
    max_dd     = drawdown.min()
    sharpe     = round(pnl.mean() / pnl.std() * np.sqrt(252), 3) if pnl.std() > 0 else 0.0
    return {'pnl': pnl, 'cumulative': cumulative, 'sharpe': sharpe,
            'max_drawdown': round(max_dd, 3), 'win_rate': win_rate, 'n_trades': n_trades}

print('Signal generator and backtest engine defined.')
print('  ✓ Stateful signal loop (correct state transitions)')
print('  ✓ Stop loss at |Z| = 4.0')
print('  ✓ P&L normalised by spread std')
print('  ✓ Per-trade win rate (not per-day)')


### 6.1 Model 1: Static OLS — Baseline

β estimated once from the full sample. Assumes the relationship is constant over 23 years. Forms the benchmark for comparison.

In [ ]:
m1 = {}
print('Model 1 — Static OLS')
for (a1,a2),info in pair_data.items():
    p   = info['df']
    sig = generate_signals(p['z_static'])
    r   = backtest(p['spread_static'], sig)
    m1[(a1,a2)] = r
    print(f"  [{info['label']}]  Sharpe={r['sharpe']}  "
          f"MaxDD={r['max_drawdown']}  WinRate={r['win_rate']}%  Trades={r['n_trades']}")


### 6.2 Model 2: Rolling OLS — Adaptive Hedge Ratio

β is re-estimated every day using only the most recent **2× half-life** days of data (pair-specific window). The rolling window is no longer a fixed 60 days for all pairs — it is calibrated to each pair's own mean-reversion speed:

| Pair | Half-life | Rolling Window |
|---|---|---|
| Energy — Crude Oil | 21.9 days | 44 days |
| Energy — Refined Products | 5.8 days | 20 days (minimum enforced) |
| Agricultural — Crush Spread | 85.8 days | 172 days |
| Agricultural — Feed Grains | 83.0 days | 166 days |

**Why this is better than a fixed 60-day window:** The window now covers approximately 2 full reversion cycles for every pair. For Gas Oil/Diesel (HL = 5.8 days), a 60-day window contained ~10 cycles — far more history than needed, making the rolling β slow to adapt. For Corn/Wheat (HL = 83 days), a 60-day window was *shorter* than one full reversion cycle, meaning the β was estimated before the spread had even finished one complete oscillation. Both extremes introduced noise. The half-life-calibrated window fixes both simultaneously.


In [ ]:
m2 = {}
print('Model 2 — Rolling OLS')
for (a1,a2),info in pair_data.items():
    p   = info['df']
    sig = generate_signals(p['z_rolling'])
    r   = backtest(p['spread_rolling'], sig)
    m2[(a1,a2)] = r
    print(f"  [{info['label']}]  Sharpe={r['sharpe']}  "
          f"MaxDD={r['max_drawdown']}  WinRate={r['win_rate']}%  Trades={r['n_trades']}")


### 6.3 Model 3: VECM — Vector Error-Correction Model

VECM models **both** series jointly, adding an error-correction term that captures how each series adjusts back to equilibrium:
$$\Delta Y_t = \alpha_1(Y_{t-1} - \hat{\beta}X_{t-1}) + \text{lagged}\;\Delta Y,\Delta X + \varepsilon_t$$

The cointegrating vector is estimated by **Maximum Likelihood** (more statistically efficient than OLS). In practice for stable pairs the improvement is marginal — OLS and MLE produce very similar β values. The `ValueWarning` about date frequency is harmless.


In [ ]:
import warnings
m3 = {}
print('Model 3 — VECM')
for (a1,a2),info in pair_data.items():
    p    = info['df'][[a1,a2]].dropna()
    roll = info['roll']  # use pair-specific half-life-based window
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')  # suppress harmless date-frequency ValueWarning
            vf  = VECM(p, k_ar_diff=2, coint_rank=1, deterministic='ci').fit()
        sv   = pd.Series(p.values @ vf.beta[:,0], index=p.index)
        mu_v = sv.rolling(roll).mean()
        sd_v = sv.rolling(roll).std()
        z_v  = (sv - mu_v) / sd_v
        sig  = generate_signals(z_v)
        r    = backtest(sv, sig)
        m3[(a1,a2)] = r
        print(f"  [{info['label']}]  Sharpe={r['sharpe']}  "
              f"MaxDD={r['max_drawdown']}  WinRate={r['win_rate']}%  Trades={r['n_trades']}")
    except Exception as e:
        print(f"  [{info['label']}] VECM failed: {e}"); m3[(a1,a2)] = None


In [ ]:
# In-sample cumulative P&L
fig, axes = plt.subplots(2,2,figsize=(18,10)); axes=axes.flatten()
for ax,((a1,a2),info) in zip(axes,pair_data.items()):
    r1=m1[(a1,a2)]; r2=m2[(a1,a2)]; r3=m3.get((a1,a2))
    r1['cumulative'].plot(ax=ax, label=f'Static OLS (SR={r1["sharpe"]})', lw=1.2)
    r2['cumulative'].plot(ax=ax, label=f'Rolling OLS (SR={r2["sharpe"]})', lw=1.0, ls='--')
    if r3: r3['cumulative'].plot(ax=ax, label=f'VECM (SR={r3["sharpe"]})', lw=1.0, ls=':')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(info['label'], fontweight='bold', fontsize=10)
    ax.set_ylabel('Cumulative P&L (normalised)'); ax.legend(fontsize=8)
plt.suptitle('In-Sample Cumulative P&L — All Pairs & Models (2000–2023)\n'
             'P&L ÷ spread σ → comparable across pairs',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


## 7. Spread Residual Diagnostics

Four plots per pair validate the key assumptions underlying the strategy:

| Plot | What It Checks | What We Want to See |
|---|---|---|
| **Spread time series + ADF** | Is the spread actually stationary? | p < 0.05 |
| **Histogram vs Normal** | Is the distribution bell-shaped? | Roughly normal — enables Z-score probability interpretation |
| **ACF** | Does autocorrelation decay exponentially? | Fast decay confirms AR(1)/OU mean-reversion structure |
| **QQ plot** | Are extreme events more common than normal predicts? | Points bowing away from diagonal = fat tails |

**Fat tails (QQ plot bowing)** are common in commodity spreads. This means ±2σ events occur more often than 5%. The stop loss at ±4σ is especially important precisely because the Gaussian model *underestimates* how often extreme events occur.


In [ ]:
fig, big_axes = plt.subplots(len(pair_data), 4, figsize=(20, 5*len(pair_data)))
for ri,((a1,a2),info) in enumerate(pair_data.items()):
    spread = info['df']['spread_static'].dropna(); axes = big_axes[ri]
    ap = adfuller(spread)[1]
    # Spread time series
    axes[0].plot(spread.index, spread, lw=0.6, color='steelblue')
    axes[0].axhline(spread.mean(), color='red', ls='--', lw=1)
    axes[0].set_title(f"{info['label']}\nADF p={ap:.4f}  "
                      f"{'✓ Stationary' if ap<0.05 else '✗ NOT stationary'}",
                      fontsize=9, fontweight='bold')
    # Histogram
    axes[1].hist(spread, bins=60, color='steelblue', edgecolor='white', density=True)
    x = np.linspace(spread.min(), spread.max(), 200)
    axes[1].plot(x, stats.norm.pdf(x, spread.mean(), spread.std()), 'r-', lw=2, label='Normal')
    axes[1].set_title('Distribution vs Normal'); axes[1].legend(fontsize=8)
    # ACF
    acf_vals = acf(spread, nlags=40, fft=True)
    axes[2].bar(range(len(acf_vals)), acf_vals, color='steelblue', width=0.6)
    axes[2].axhline(1.96/np.sqrt(len(spread)), color='red', ls='--', lw=1)
    axes[2].axhline(-1.96/np.sqrt(len(spread)), color='red', ls='--', lw=1)
    axes[2].set_title('ACF — should decay exponentially'); axes[2].set_xlabel('Lag (days)')
    # QQ plot
    stats.probplot(spread, dist='norm', plot=axes[3])
    axes[3].set_title('QQ Plot — deviation from line = fat tails')
plt.suptitle('Spread Residual Diagnostics', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()


## 8. Walk-Forward Cross-Validation

### Why Standard K-Fold CV Fails on Time Series
Standard cross-validation randomly shuffles data into k folds. For time series this is **catastrophically wrong**: training on 2022 data to predict 2018 uses information that did not exist yet — look-ahead bias. The resulting Sharpe ratios are fictionally high and completely non-deployable.

### Walk-Forward (Expanding Window) Approach
```
Fold 1: [── TRAIN: 2000→2018 ──]  [TEST: 2019]
Fold 2: [──── TRAIN: 2000→2019 ────]  [TEST: 2020]
Fold 3: [────── TRAIN: 2000→2020 ──────]  [TEST: 2021]
Fold 4: [──────── TRAIN: 2000→2021 ────────]  [TEST: 2022]
Fold 5: [────────── TRAIN: 2000→2022 ──────────]  [TEST: 2023]
```

At each fold: (1) estimate β, μ, σ on training only — parameters are then **frozen**; (2) generate signals and compute P&L on the unseen test window; (3) record Sharpe, trades, win rate. Time ordering is always preserved.

### Zero-Trade Folds
When a fold shows 0 trades, the Z-score never breached ±2σ in that calendar year. This is **correct behaviour** — the strategy should stay flat when there is no signal. For WTI/Brent, the 2022–2023 period had unusually compressed differentials (Russian oil sanctions); the strategy correctly took no position.


In [ ]:
def walk_forward_cv(p, a1, a2, roll, n_folds=5, test_size=252):
    """
    Walk-forward CV using pair-specific rolling window.
    roll: half-life-based window for Z-score normalisation (passed from pair_data).
    All parameters (β, rolling mean, rolling std) are estimated on training data only.
    """
    total = len(p); min_train = max(3*252, 3*roll)  # min_train also respects window size
    start_test = total - n_folds*test_size
    if start_test < min_train:
        n_folds = max(1, (total-min_train)//test_size)
        start_test = total - n_folds*test_size
    rows = []
    for fold in range(n_folds):
        t0 = start_test + fold*test_size
        t1 = min(t0 + test_size, total)
        train = p.iloc[:t0]; test = p.iloc[t0:t1]
        # Estimate all parameters on TRAINING data only
        b     = sm.OLS(train[a2], sm.add_constant(train[a1])).fit().params[a1]
        sp_tr = train[a2] - b*train[a1]
        # Use rolling stats from training window to get the 'current' mean and std
        # (last value of the rolling mean/std over training period)
        mu_tr = sp_tr.rolling(roll).mean().iloc[-1]
        std_tr = sp_tr.rolling(roll).std().iloc[-1]
        if np.isnan(mu_tr) or np.isnan(std_tr) or std_tr <= 0:
            mu_tr, std_tr = sp_tr.mean(), sp_tr.std()  # fallback to full-period stats
        # Apply FROZEN parameters to test window
        sp_te = test[a2] - b*test[a1]
        z_te  = (sp_te - mu_tr) / std_tr
        sig   = generate_signals(z_te)
        bt    = backtest(sp_te, sig, spread_std=std_tr)
        rows.append({'Fold': fold+1,
                     'Train': f'{p.index[0].date()}→{p.index[t0-1].date()}',
                     'Test':  f'{p.index[t0].date()}→{p.index[t1-1].date()}',
                     'Sharpe': bt['sharpe'], 'MaxDD': bt['max_drawdown'],
                     'WinRate%': bt['win_rate'], 'Trades': bt['n_trades']})
    return pd.DataFrame(rows)

cv_results = {}
print('Walk-Forward Cross-Validation Results')
print('='*70)
for (a1,a2),info in pair_data.items():
    p    = info['df'][[a1,a2]].dropna()
    roll = info['roll']  # pair-specific window
    cv   = walk_forward_cv(p, a1, a2, roll)
    cv_results[(a1,a2)] = cv
    active = cv[cv['Trades']>0]
    print(f"\n[{info['label']}]  (roll window = {roll} days)")
    print(cv.to_string(index=False))
    if len(active) >= 2:
        print(f"  Active folds: {len(active)}/{len(cv)}  "
              f"Mean Sharpe={active['Sharpe'].mean():.3f}  "
              f"Std={active['Sharpe'].std():.3f}")
    elif len(active) == 1:
        print(f"  Active folds: 1/{len(cv)}  Mean Sharpe={active['Sharpe'].mean():.3f}  Std=N/A")
    else:
        print(f"  Active folds: 0/{len(cv)} — no signals generated")

In [ ]:
fig, axes = plt.subplots(2,2,figsize=(16,9)); axes=axes.flatten()
for ax,((a1,a2),cv) in zip(axes,cv_results.items()):
    label = pair_data[(a1,a2)]['label']
    colors = [('green' if s>0 else 'red') if n>0 else 'lightgray'
              for s,n in zip(cv['Sharpe'],cv['Trades'])]
    ax.bar(cv['Fold'], cv['Sharpe'], color=colors)
    ax.axhline(0, color='black', lw=0.8)
    ax.axhline(cv['Sharpe'].mean(), color='blue', ls='--', lw=1.2,
               label=f'Mean={cv["Sharpe"].mean():.2f}')
    ax.set_title(label, fontweight='bold', fontsize=10)
    ax.set_xlabel('Fold'); ax.set_ylabel('OOS Sharpe Ratio')
    ax.legend(fontsize=8)
    for _,row in cv.iterrows():
        if row['Trades'] == 0:
            ax.text(row['Fold'], 0.02, '0 trades\n(stable)', ha='center',
                    va='bottom', fontsize=7, color='gray')
plt.suptitle('Walk-Forward CV — Out-of-Sample Sharpe per Fold\n'
             'Green = profitable  Red = loss  Gray = no trades (spread too stable to signal)',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

## 9. Out-of-Sample Backtest (Train: 2000–2019 / Test: 2020–2023)

### The Strictest Validation
All parameters estimated on 2000–2019, then completely **frozen**. No re-estimation. The 2020–2023 test period includes:
- **March 2020:** COVID crash — WTI briefly went negative (April 20, 2020)
- **2021:** Commodity supercycle recovery — large directional moves
- **2022:** Russia-Ukraine war — energy market restructuring; sanctions compressed WTI-Brent differential
- **2023:** Gradual normalisation — tighter spreads, fewer signals

### Expected Finding: Gas Oil/Diesel OOS Sharpe = −0.801
This is **not a code error** — it is an economically important and honest result:

- **Why it fails OOS:** Gas Oil/Diesel has the shortest half-life (~5.8 days), generating ~47 OOS trades. The 2022 Russia-Ukraine shock structurally elevated European Gas Oil premiums, breaking the 2000–2019 equilibrium the model was trained on. Every trade bet on reversion to a mean that had shifted.
- **Why this is valuable:** It shows that even the strongest in-sample pair (IS Sharpe = 1.664) can fail when market structure changes. This is precisely why we run OOS tests.
- **The agricultural pairs (Soybeans/Meal OOS Sharpe = +0.913) outperform OOS** because their mean-reversion mechanism — crush economics and feed-grain substitution — is physically enforced by supply chains regardless of geopolitics.

We test two entry thresholds: **±2.0** (conservative) and **±1.5** (more active).

In [ ]:
TRAIN_END = '2019-12-31'
OOS = {}
print(f'Train: 2000-01-01 → {TRAIN_END}')
print(f'Test:  2020-01-01 → 2023-08-04\n{"="*62}')
for (a1,a2),info in pair_data.items():
    p     = info['df'][[a1,a2]].dropna()
    train = p.loc[:TRAIN_END]; test = p.loc['2020-01-01':]
    b     = sm.OLS(train[a2], sm.add_constant(train[a1])).fit().params[a1]
    sp_tr = train[a2] - b*train[a1]
    mu_tr, std_tr = sp_tr.mean(), sp_tr.std()
    sp_te = test[a2] - b*test[a1]
    z_te  = (sp_te - mu_tr) / std_tr
    s20   = generate_signals(z_te, entry=2.0); b20 = backtest(sp_te, s20, spread_std=std_tr)
    s15   = generate_signals(z_te, entry=1.5); b15 = backtest(sp_te, s15, spread_std=std_tr)
    OOS[(a1,a2)] = {'info':info,'sp_te':sp_te,'z_te':z_te,
                    'b20':b20,'s20':s20,'b15':b15,'s15':s15,
                    'beta':round(b,4),'mu_tr':mu_tr,'std_tr':std_tr}
    print(f"\n[{info['label']}]")
    print(f"  ±2.0 → Sharpe={b20['sharpe']}  MaxDD={b20['max_drawdown']}  "
          f"WinRate={b20['win_rate']}%  Trades={b20['n_trades']}")
    print(f"  ±1.5 → Sharpe={b15['sharpe']}  MaxDD={b15['max_drawdown']}  "
          f"WinRate={b15['win_rate']}%  Trades={b15['n_trades']}")

In [ ]:
fig, axes = plt.subplots(2,2,figsize=(18,10)); axes=axes.flatten()
for ax,((a1,a2),res) in zip(axes,OOS.items()):
    b20=res['b20']; b15=res['b15']; label=res['info']['label']
    b20['cumulative'].plot(ax=ax, color='steelblue', lw=1.3,
                          label=f'Entry ±2.0 (SR={b20["sharpe"]})')
    b15['cumulative'].plot(ax=ax, color='darkorange', lw=1.0, ls='--',
                          label=f'Entry ±1.5 (SR={b15["sharpe"]})')
    ax.axhline(0, color='black', lw=0.8)
    cum = b20['cumulative']
    ax.fill_between(cum.index, 0, cum, where=cum>=0, alpha=0.10, color='green')
    ax.fill_between(cum.index, 0, cum, where=cum<0,  alpha=0.10, color='red')
    ax.set_title(label, fontweight='bold'); ax.set_ylabel('Cum. P&L (normalised)')
    ax.legend(fontsize=9)
plt.suptitle('OOS Cumulative P&L (2020–2023) — Parameters Frozen from 2000–2019 Training',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Detailed OOS view for best pair
best_pair = ('LOW SULPHUR GAS OIL', 'ULS DIESEL')
res = OOS[best_pair]; sp=res['sp_te']; z=res['z_te']; sig=res['s20']; bt=res['b20']
fig, axes = plt.subplots(3,1,figsize=(18,12), sharex=True)
fig.suptitle(f'OOS Detail: {best_pair[0]} / {best_pair[1]} (2020–2023)',
             fontsize=13, fontweight='bold')
axes[0].plot(sp.index, sp, color='purple', lw=0.8, label='Spread')
axes[0].axhline(res['mu_tr'], color='black', ls='--', lw=1, label='Train μ')
li=sig[sig==1].index; si=sig[sig==-1].index
axes[0].scatter(li, sp.loc[li], color='green',  marker='^', s=30, label='Long entry',  zorder=5)
axes[0].scatter(si, sp.loc[si], color='red',    marker='v', s=30, label='Short entry', zorder=5)
axes[0].set_ylabel('Spread'); axes[0].legend(fontsize=8)
axes[1].plot(z.index, z, color='darkorange', lw=0.8)
for th,c,ls in [(2,'red','--'),(-2,'green','--'),(0.5,'gray',':'),(-0.5,'gray',':'),(4,'darkred','-'),(-4,'darkred','-')]:
    axes[1].axhline(th, color=c, ls=ls, lw=0.8)
axes[1].set_ylabel('Z-Score')
axes[1].set_title('Z-Score (dashed red=±2 entry | dotted=±0.5 exit | solid dark red=±4 stop loss)')
bt['cumulative'].plot(ax=axes[2], color='steelblue', lw=1.5)
axes[2].axhline(0, color='black', lw=0.8); axes[2].set_ylabel('Cumulative P&L')
axes[2].set_title(f'Cumulative P&L  SR={bt["sharpe"]}  MaxDD={bt["max_drawdown"]}  '
                  f'WinRate={bt["win_rate"]}%  Trades={bt["n_trades"]}')
plt.tight_layout(); plt.show()

## 10. Model Comparison

### Ranking Framework
The **primary** ranking criterion is OOS Sharpe (2020–2023) — this is the only metric using genuinely unseen data. In-sample Sharpe is always inflated because the model fitted the same data it's evaluated on. Secondary criteria are CV stability (consistency across regimes) and maximum drawdown.


In [ ]:
rows = []
for (a1,a2),info in pair_data.items():
    cv=cv_results[(a1,a2)]; oos_r=OOS[(a1,a2)]
    for mname,mres in [('Static OLS',m1),('Rolling OLS',m2),('VECM',m3)]:
        r = mres.get((a1,a2))
        if r is None: continue
        rows.append({'Pair':info['label'],'Model':mname,
                     'IS Sharpe':r['sharpe'],
                     'OOS Sharpe':oos_r['b20']['sharpe'],
                     'CV Mean':round(cv['Sharpe'].mean(),3),
                     'MaxDD':r['max_drawdown'],
                     'WinRate%':r['win_rate'],
                     'Trades':r['n_trades']})
comp = pd.DataFrame(rows)
print(comp.to_string(index=False))

In [ ]:
pivot = comp.pivot_table(index='Pair', columns='Model', values='IS Sharpe', aggfunc='first')
fig, axes = plt.subplots(1,2,figsize=(16,5))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
            linewidths=0.5, annot_kws={'size':11}, ax=axes[0])
axes[0].set_title('In-Sample Sharpe (2000–2023)', fontweight='bold')
oos_s = pd.DataFrame([{'Pair':info['label'],
    'OOS Sharpe': OOS[(a1,a2)]['b20']['sharpe'],
    'CV Mean Sharpe': round(cv_results[(a1,a2)]['Sharpe'].mean(),3),
    'OOS MaxDD': OOS[(a1,a2)]['b20']['max_drawdown']}
    for (a1,a2),info in pair_data.items()]).set_index('Pair')
sns.heatmap(oos_s, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
            linewidths=0.5, annot_kws={'size':11}, ax=axes[1])
axes[1].set_title('OOS & CV Performance (test: 2020–2023)', fontweight='bold')
plt.suptitle('Model Comparison — Primary metric is OOS Sharpe (rightmost heatmap)',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
print('='*72)
print('FINAL RESULTS SUMMARY')
print('='*72)
for (a1,a2),info in pair_data.items():
    oos=OOS[(a1,a2)]; cv=cv_results[(a1,a2)]; bt=oos['b20']
    active=cv[cv['Trades']>0]
    if len(active) >= 2:
        cv_s = (f"{active['Sharpe'].mean():.3f}±{active['Sharpe'].std():.3f} "
                f"({len(active)}/{len(cv)} active folds)")
    elif len(active) == 1:
        cv_s = f"{active['Sharpe'].mean():.3f} (1/{len(cv)} active fold — std N/A)"
    else:
        cv_s = 'No active folds'
    print(f"\n[{info['label']}]  {a1} / {a2}")
    print(f"  Hedge Ratio β:    {info['beta']}")
    print(f"  Half-life:        {info['half_life']} trading days")
    print(f"  Rolling Window:   {info['roll']} days  (= 2 × {info['half_life']} HL, min 20)")
    print(f"  IS Sharpe:        {m1[(a1,a2)]['sharpe']} (Static OLS)")
    print(f"  OOS Sharpe:       {bt['sharpe']}  MaxDD={bt['max_drawdown']}  "
          f"WinRate={bt['win_rate']}%  Trades={bt['n_trades']}")
    print(f"  Walk-forward CV:  {cv_s}")

## 11. Granger Causality — Who Leads Whom?

### What Granger Causality Tests
Granger causality is **not** true causation. It asks: *does knowing the past values of series A improve my ability to forecast series B, beyond what B's own past tells me?* If yes, A **Granger-causes** B — it moves first and B follows with a lag.

### The Test Mechanics
Two VAR models are compared for each direction A→B:

- **Restricted** (B's own lags only): $B_t = \alpha + \sum \beta_k B_{t-k} + \varepsilon_t$
- **Unrestricted** (B's own lags + A's lags): $B_t = \alpha + \sum \beta_k B_{t-k} + \sum \gamma_k A_{t-k} + \varepsilon_t$

An F-test checks whether adding A's lags significantly reduces residual variance. **H₀: A does NOT Granger-cause B** (all γ_k = 0). Reject H₀ (p < 0.05) → A leads B.

### Why First Differences?
Granger causality requires stationary input. Running it on price *levels* (I(1) non-stationary) violates the F-test assumptions and produces spurious results. We use **daily returns** (first differences).

### Trading Implication
If A one-way leads B at lag 1, then when A makes a large move that pushes the spread away from equilibrium, we can enter the pairs trade *immediately*, before B has had time to adjust. This improves entry price timing.


In [ ]:
LAGS_TO_TEST = [1, 2, 5, 10]  # 1 day, 2 days, 1 week, 2 weeks
granger_results = {}

for a1,a2,label in PAIRS:
    p  = pair_data[(a1,a2)]['prices'] if 'prices' in pair_data[(a1,a2)] \
         else pair_data[(a1,a2)]['df'][[a1,a2]]
    dp = p[[a1,a2]].diff().dropna()  # first differences — required for stationarity
    results_pair = {}
    for direction, (y_col, x_col) in [('A->B', (a2,a1)), ('B->A', (a1,a2))]:
        data = dp[[y_col, x_col]]
        gc   = grangercausalitytests(data, maxlag=max(LAGS_TO_TEST), verbose=False)
        lag_results = {}
        for lag in LAGS_TO_TEST:
            f_stat = gc[lag][0]['ssr_ftest'][0]
            p_val  = gc[lag][0]['ssr_ftest'][1]
            lag_results[lag] = {'f': round(f_stat,3), 'p': round(p_val,4)}
        results_pair[direction] = lag_results
    granger_results[(a1,a2)] = results_pair

print('Granger Causality Results  (p-values | * p<0.05  ** p<0.01  *** p<0.001)')
print('='*78)
for a1,a2,label in PAIRS:
    res = granger_results[(a1,a2)]
    print(f'\n{label}: {a1} → {a2}')
    print(f'  {"Direction":<30} {"Lag-1":>10} {"Lag-2":>10} {"Lag-5":>10} {"Lag-10":>10}')
    print(f'  {"-"*65}')
    for direction, name in [('A->B', f'{a1}→{a2}'), ('B->A', f'{a2}→{a1}')]:
        row = f'  {name[:28]:<28}'
        for lag in LAGS_TO_TEST:
            p = res[direction][lag]['p']
            stars = '***' if p<0.001 else ('**' if p<0.01 else ('* ' if p<0.05 else '  '))
            row += f'  {p:.4f}{stars}'
        print(row)
    # Nuanced interpretation: detect dominant direction
    fwd_lags = [l for l in LAGS_TO_TEST if res['A->B'][l]['p'] < 0.05]
    bwd_lags = [l for l in LAGS_TO_TEST if res['B->A'][l]['p'] < 0.05]
    fwd = len(fwd_lags) > 0; bwd = len(bwd_lags) > 0
    if fwd and bwd:
        fwd_min = min(fwd_lags); bwd_min = min(bwd_lags)
        # One direction dominant if: shorter min lag AND more significant lags
        if fwd_min < bwd_min and len(fwd_lags) > len(bwd_lags):
            interp = f'→ {a1} leads {a2} (dominant from lag {fwd_min}); {a2}→{a1} only at lag {bwd_min}+'
        elif bwd_min < fwd_min and len(bwd_lags) > len(fwd_lags):
            interp = f'→ {a2} leads {a1} (dominant from lag {bwd_min}); {a1}→{a2} only at lag {fwd_min}+'
        else:
            interp = f'↔ Bidirectional — both lead from lag {min(fwd_min, bwd_min)}'
    elif fwd:
        interp = f'→ {a1} is the PRICE LEADER — {a2} follows from lag {min(fwd_lags)}'
    elif bwd:
        interp = f'→ {a2} is the PRICE LEADER — {a1} follows from lag {min(bwd_lags)}'
    else:
        interp = '— No Granger causality (contemporaneous comovement only)'
    print(f'  Interpretation: {interp}')


In [ ]:
# Heatmap of p-values — green = significant (price leadership exists)
fig, axes = plt.subplots(2,2,figsize=(16,10)); axes=axes.flatten()
for ax,(a1,a2,label) in zip(axes,PAIRS):
    res = granger_results[(a1,a2)]
    dirs = [f'{a1}\n→{a2}', f'{a2}\n→{a1}']
    matrix = np.array([[res['A->B'][l]['p'] for l in LAGS_TO_TEST],
                       [res['B->A'][l]['p'] for l in LAGS_TO_TEST]])
    mat_df = pd.DataFrame(matrix, index=dirs, columns=[f'Lag {l}' for l in LAGS_TO_TEST])
    sns.heatmap(mat_df, annot=True, fmt='.4f', ax=ax,
                cmap='RdYlGn_r', vmin=0, vmax=0.1,
                linewidths=0.5, cbar=False)
    ax.set_title(f'{label}\np-values (green = significant price leadership, p<0.05)',
                 fontsize=9, fontweight='bold')
plt.suptitle('Granger Causality — Who Leads Whom? (tested on first-differenced prices)',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

## 12. OU Forecasting — Train on 2000→Feb 2023, Test on Last 6 Months

### The Setup
We fit the Ornstein-Uhlenbeck model on all data **except** the last 6 months, then forecast forward and compare against what actually happened.

$$dS_t = \kappa(\theta - S_t)\,dt + \sigma\,dW_t$$

| Parameter | Meaning |
|---|---|
| **κ** (kappa) | Reversion speed — how strongly the spread is pulled toward θ. κ=0 → random walk |
| **θ** (theta) | Long-run mean — equilibrium the spread reverts to |
| **σ** (sigma) | Daily volatility — determines width of confidence bands |
| **Half-life = ln(2)/κ** | Expected days to close half the gap to θ |

### Estimation: AR(1) → OU Parameters
Discretise: $S_t = \alpha + \phi S_{t-1} + \varepsilon_t$. OLS gives $\hat{\phi}$, $\hat{\alpha}$. Then:
$\kappa = -\ln(\phi)$, $\theta = \alpha/(1-\phi)$, $\sigma = \text{std}(\varepsilon_t)$.

### Accuracy Metrics
| Metric | Ideal | If Wrong |
|---|---|---|
| **50% CI coverage** | ~50% of actual values inside band | Too low → model underestimates variance, or regime broke |
| **90% CI coverage** | ~90% | Too low → extreme regime — OU assumptions violated |
| **Median RMSE** | Small relative to spread std | Large → spread took unexpected path |
| **Convergence accuracy** | Actual ≈ predicted date | Gap → mean shifted or reversion speed changed |

### Regime-Break Pairs: When Low Coverage is the Point, Not a Bug
**Gas Oil/Diesel** (50% CI coverage = 2.3%): S₀ = 20.82, θ = 4.64 — model predicted 99.5% of paths converge by Feb 20. But the actual spread *never crossed θ* during the entire 6-month test window. This is a regime break: the 2022 Russia-Ukraine shock structurally elevated the Gas Oil premium throughout early 2023, violating the OU mean-reversion assumption. The 2.3% coverage is not a coding error — it is the model correctly diagnosing that the regime had shifted. **This is consistent with the negative OOS backtest Sharpe for this pair.**

**Corn/Wheat** (RMSE = 67.5 on a spread with std ~11): S₀ = 65.5, θ = 119.75 — spread was 54 units *below* mean because the 2022 Ukraine war made wheat plentiful and corn expensive (corn premium). The OU model expected fast convergence back to the historical equilibrium; the actual spread moved slowly and in unexpected directions. Again, this is a genuine macro regime finding.


In [ ]:
def fit_ou(spread):
    """Estimate OU params via AR(1) OLS on the spread series."""
    s   = spread.dropna()
    lag = s.shift(1).dropna(); curr = s.loc[lag.index]
    res = sm.OLS(curr, sm.add_constant(lag)).fit()
    phi   = float(res.params.iloc[1])
    alpha = float(res.params['const'])
    phi   = max(min(phi, 0.9999), 0.0001)  # must stay in (0,1) for mean reversion
    kappa = -np.log(phi)
    theta = alpha / (1-phi) if abs(1-phi) > 1e-6 else float(s.mean())
    sigma = float(np.sqrt(res.mse_resid))
    hl    = np.log(2) / kappa if kappa > 0 else np.nan
    return {'kappa':kappa,'theta':theta,'sigma':sigma,'half_life':hl}

def simulate_ou(params, S0, n_steps, n_paths=5000):
    """Euler-Maruyama: S_{t+1} = S_t + κ(θ - S_t) + σ*ε_t"""
    k,th,sig = params['kappa'], params['theta'], params['sigma']
    paths    = np.zeros((n_steps+1, n_paths)); paths[0] = S0
    shocks   = np.random.randn(n_steps, n_paths)
    for t in range(1, n_steps+1):
        paths[t] = paths[t-1] + k*(th - paths[t-1]) + sig*shocks[t-1]
    return paths

print('OU helper functions defined.')

In [ ]:
# Train/test split: everything except last 6 months
max_date = df_clean.index.max()
CUTOFF   = max_date - pd.DateOffset(months=6)
print(f'Train: {df_clean.index.min().date()} → {CUTOFF.date()}')
print(f'Test:  {CUTOFF.date()} → {max_date.date()}  (~130 trading days)\n')

ou_tt = {}  # train/test OU results

for a1,a2,label in PAIRS:
    p    = pair_data[(a1,a2)]['df'][[a1,a2]].dropna()
    train = p.loc[:CUTOFF]; test = p.loc[CUTOFF:]
    # Estimate beta on TRAINING only
    b_tr         = sm.OLS(train[a2], sm.add_constant(train[a1])).fit().params[a1]
    spread_train = train[a2] - b_tr*train[a1]
    spread_test  = test[a2]  - b_tr*test[a1]
    # Fit OU on TRAINING spread only
    ou = fit_ou(spread_train)
    S0 = float(spread_train.iloc[-1])
    n_test = len(spread_test)
    # Simulate forward for exactly n_test steps
    paths = simulate_ou(ou, S0, n_steps=n_test)
    fd    = pd.bdate_range(start=spread_test.index[0], periods=n_test+1)
    pcts  = {q: np.percentile(paths, q, axis=1) for q in [5,25,50,75,95]}
    # Coverage metrics
    act   = spread_test.values
    n_al  = min(len(act), n_test)
    in_50 = np.mean((act[:n_al]>=pcts[25][1:n_al+1]) & (act[:n_al]<=pcts[75][1:n_al+1]))*100
    in_90 = np.mean((act[:n_al]>=pcts[5][1:n_al+1])  & (act[:n_al]<=pcts[95][1:n_al+1]))*100
    rmse  = np.sqrt(np.mean((pcts[50][1:n_al+1] - act[:n_al])**2))
    # Did actual spread cross theta?
    theta = ou['theta']
    direction = 1 if S0 > theta else -1
    crossings = spread_test[spread_test<=theta] if direction==1 else spread_test[spread_test>=theta]
    actual_conv = crossings.index[0] if len(crossings) else None
    # Predicted convergence (median first-crossing across simulated paths)
    sim_cross = []
    for path in paths.T:
        idx = np.where(path<=theta)[0] if direction==1 else np.where(path>=theta)[0]
        if len(idx): sim_cross.append(idx[0])
    pct_conv = len(sim_cross)/paths.shape[1]*100
    pred_conv = fd[int(np.median(sim_cross))] if sim_cross else None
    ou_tt[(a1,a2)] = {'label':label,'ou':ou,'S0':S0,'spread_train':spread_train,
                      'spread_test':spread_test,'paths':paths,'pcts':pcts,'fd':fd,
                      'in_50':in_50,'in_90':in_90,'rmse':rmse,
                      'actual_conv':actual_conv,'pred_conv':pred_conv,'pct_conv':pct_conv}
    print(f'[{label}]')
    print(f'  OU: κ={ou["kappa"]:.4f}  θ={ou["theta"]:.2f}  σ={ou["sigma"]:.4f}  '
          f'HL={ou["half_life"]:.1f}d')
    print(f'  S₀={S0:.3f}  θ={theta:.3f}  '
          f'({"above" if S0>theta else "below"} by {abs(S0-theta):.3f})')
    print(f'  Predicted conv: {pred_conv.date() if pred_conv else "N/A"}  '
          f'({pct_conv:.1f}% of paths converge within test window)')
    print(f'  Actual conv:    {actual_conv.date() if actual_conv else "Did not cross θ"}')
    print(f'  50% CI coverage: {in_50:.1f}%  |  90% CI coverage: {in_90:.1f}%  '
          f'|  RMSE: {rmse:.4f}\n')

In [ ]:
# Fan charts with actual test spread overlaid in red
fig, axes = plt.subplots(2,2,figsize=(20,14)); axes=axes.flatten()
for ax,(a1,a2,label) in zip(axes,PAIRS):
    r   = ou_tt[(a1,a2)]; ou=r['ou']; pcts=r['pcts']; fd=r['fd']
    # Historical training spread (last 2 years for readability)
    hist = r['spread_train'].loc['2021':]
    ax.plot(hist.index, hist.values, color='black', lw=1.0,
            label='Training spread', zorder=5)
    # Confidence bands
    ax.fill_between(fd, pcts[5],  pcts[95], alpha=0.12, color='royalblue', label='90% CI')
    ax.fill_between(fd, pcts[25], pcts[75], alpha=0.25, color='royalblue', label='50% CI')
    ax.plot(fd, pcts[50], color='royalblue', lw=1.8, ls='--', label='Median forecast')
    # Long-run mean
    ax.axhline(ou['theta'], color='green', ls=':', lw=1.5,
               label=f"θ = {ou['theta']:.2f}")
    # Train/test boundary
    ax.axvline(CUTOFF, color='gray', ls=':', lw=1.0, label='Train | Test')
    # S0 marker
    ax.scatter([fd[0]], [r['S0']], color='royalblue', s=80, zorder=6)
    # ACTUAL test spread — red
    ax.plot(r['spread_test'].index, r['spread_test'].values,
            color='crimson', lw=2.2, label='Actual (test period)', zorder=7)
    # Convergence lines
    if r['pred_conv']:
        ax.axvline(r['pred_conv'], color='orange', ls='--', lw=1.5,
                   label=f"Pred. conv. {r['pred_conv'].strftime('%b %d')}")
    if r['actual_conv']:
        ax.axvline(r['actual_conv'], color='crimson', ls='-', lw=2.0,
                   label=f"Actual conv. {r['actual_conv'].strftime('%b %d')}")
    ax.set_title(f"{label}\nκ={ou['kappa']:.4f}  θ={ou['theta']:.2f}  "
                 f"HL={ou['half_life']:.1f}d  |  "
                 f"50%cov={r['in_50']:.0f}%  90%cov={r['in_90']:.0f}%  "
                 f"RMSE={r['rmse']:.3f}",
                 fontsize=9, fontweight='bold')
    ax.set_ylabel('Spread'); ax.legend(fontsize=7, loc='best')
plt.suptitle('OU Forecast vs Actual — Train: Jan 2000→Feb 2023 | Test: Feb 2023→Aug 2023\n'
             'Blue fan = forecast confidence bands  |  Red = what actually happened',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Convergence-time distribution for best pair
bp  = ('LOW SULPHUR GAS OIL','ULS DIESEL')
r   = ou_tt[bp]; ou=r['ou']; fd=r['fd']; pcts=r['pcts']
sim_cross = []
theta = ou['theta']; S0 = r['S0']; direction = 1 if S0>theta else -1
for path in r['paths'].T:
    idx = np.where(path<=theta)[0] if direction==1 else np.where(path>=theta)[0]
    if len(idx): sim_cross.append(idx[0])
fig, axes = plt.subplots(1,2,figsize=(18,6))
fig.suptitle(f'OU Detail: {bp[0]} / {bp[1]}', fontsize=12, fontweight='bold')
# Left: zoomed fan chart on test window only
ax = axes[0]
ax.fill_between(fd, pcts[5],  pcts[95], alpha=0.12, color='royalblue', label='90% CI')
ax.fill_between(fd, pcts[25], pcts[75], alpha=0.25, color='royalblue', label='50% CI')
ax.plot(fd, pcts[50], color='royalblue', lw=2.0, ls='--', label='Median forecast')
ax.plot(r['spread_test'].index, r['spread_test'].values,
        color='crimson', lw=2.5, label='Actual', zorder=5)
ax.axhline(theta, color='green', ls=':', lw=1.5, label=f'θ={theta:.2f}')
ax.scatter([fd[0]], [S0], color='black', s=100, zorder=6)
if r['pred_conv']:   ax.axvline(r['pred_conv'],   color='orange',  ls='--', lw=2)
if r['actual_conv']: ax.axvline(r['actual_conv'], color='crimson', ls='-',  lw=2.5)
ax.set_title('Forecast Window (test period only)'); ax.legend(fontsize=8)
# Right: distribution of simulated first-crossing times
ax = axes[1]
if sim_cross:
    ax.hist(sim_cross, bins=40, color='royalblue', edgecolor='white',
            density=True, alpha=0.7, label='Sim. crossing days')
    ax.axvline(np.median(sim_cross),      color='orange',  lw=2, ls='--',
               label=f'Median day {int(np.median(sim_cross))}')
    ax.axvline(np.percentile(sim_cross,25), color='green', lw=1.5, ls=':')
    ax.axvline(np.percentile(sim_cross,75), color='red',   lw=1.5, ls=':')
    if r['actual_conv']:
        actual_day = len(r['spread_test'].loc[:r['actual_conv']])
        ax.axvline(actual_day, color='crimson', lw=3,
                   label=f'Actual: day ~{actual_day}')
ax.set_xlabel('Trading days until spread crosses θ')
ax.set_title(f'Distribution of First-Crossing Times\n'
             f'{r["pct_conv"]:.1f}% of 5,000 paths converged within test window')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 13. OU Forecasting — Full Forward Prediction (Aug 2023 → 2025)

### Fitting on Full History
Here we fit OU on the **entire 2000–2023 history** (not just a sub-period) to get robust long-run parameters. This matters because fitting on only 2020–2023 gave a 2-day half-life (unusually tight/stable regime), making the model predict convergence trivially 2 days after the dataset ends.

Full-history fitting captures the representative long-run κ across all regimes — bull markets, bear markets, crises, and calm periods.

We simulate 5,000 paths from the last observed spread value forward for 504 trading days (~2 years). The optional yfinance cell overlays actual 2023–2026 prices to verify the prediction.

In [ ]:
ou_fc = {}; future_dates = pd.bdate_range(start='2023-08-07', periods=505)

print('OU Parameters — full history 2000–2023:')
print(f'{"Pair":<35} {"HL":>8} {"θ":>8} {"S₀":>8}')
print('-'*62)
for (a1,a2),info in pair_data.items():
    spread_full = info['df']['spread_static'].dropna()
    ou   = fit_ou(spread_full)   # full history — robust long-run parameters
    S0   = float(spread_full.iloc[-1])
    paths= simulate_ou(ou, S0, n_steps=504, n_paths=5000)
    pcts = {q: np.percentile(paths, q, axis=1) for q in [5,25,50,75,95]}
    # Convergence
    theta = ou['theta']; direction = 1 if S0>theta else -1
    crossing = []
    for path in paths.T:
        idx = np.where(path<=theta)[0] if direction==1 else np.where(path>=theta)[0]
        if len(idx): crossing.append(idx[0])
    pct_conv = len(crossing)/paths.shape[1]*100
    med_date = future_dates[int(np.median(crossing))] if crossing else None
    ou_fc[(a1,a2)] = {'ou':ou,'S0':S0,'pcts':pcts,'crossing':crossing,
                      'pct_conv':pct_conv,'med_date':med_date,'theta':theta}
    print(f'{info["label"]:<35} {ou["half_life"]:>8.1f} {theta:>8.2f} {S0:>8.3f}')

In [ ]:
fig, axes = plt.subplots(2,2,figsize=(20,12)); axes=axes.flatten()
for ax,((a1,a2),info) in zip(axes,pair_data.items()):
    fc  = ou_fc[(a1,a2)]; pcts=fc['pcts']
    hist= info['df']['spread_static'].loc['2021':]
    ax.plot(hist.index, hist.values, color='black', lw=1.2, label='Historical spread')
    ax.fill_between(future_dates, pcts[5],  pcts[95], alpha=0.10, color='royalblue', label='90% CI')
    ax.fill_between(future_dates, pcts[25], pcts[75], alpha=0.22, color='royalblue', label='50% CI')
    ax.plot(future_dates, pcts[50], color='royalblue', lw=1.5, ls='--', label='Median')
    ax.axhline(fc['theta'], color='green', ls=':', lw=1.2, label=f"θ={fc['theta']:.1f}")
    if fc['med_date']:
        ax.axvline(fc['med_date'], color='orange', ls='--', lw=1.5,
                   label=f"Med. conv. {fc['med_date'].strftime('%b %Y')}")
    ou  = fc['ou']
    hl_str = f"HL={ou['half_life']:.1f}d" if ou['half_life'] < 15 else f"HL={ou['half_life']:.0f}d"
    ax.set_title(f"{info['label']}\n"
                 f"κ={ou['kappa']:.4f}  θ={ou['theta']:.2f}  {hl_str}  "
                 f"({fc['pct_conv']:.1f}% paths converge within 2 yrs)",
                 fontsize=9, fontweight='bold')
    ax.set_ylabel('Spread'); ax.legend(fontsize=7)
plt.suptitle('OU Forward Forecast from Aug 2023 (fitted on full 2000–2023 history)\n'
             'Orange dashed = predicted median convergence date',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

### 13.1 Verification Against Actual Post-2023 Data
Run the cell below to download actual 2023–2026 prices via `yfinance` and overlay them on the fan chart. This verifies whether the predicted convergence dates were accurate.

In [ ]:
try:
    import yfinance as yf
    TICKERS = {
        ('WTI CRUDE','BRENT CRUDE'): ('CL=F','BZ=F'),
        ('CORN','WHEAT'):            ('ZC=F','ZW=F'),
    }
    for (a1,a2),(t1,t2) in TICKERS.items():
        if (a1,a2) not in pair_data: continue
        try:
            p1 = yf.download(t1,start='2022-01-01',end='2026-03-01',progress=False)['Close'].squeeze()
            p2 = yf.download(t2,start='2022-01-01',end='2026-03-01',progress=False)['Close'].squeeze()
            act = pd.DataFrame({'p1':p1,'p2':p2}).dropna()
            beta_is  = pair_data[(a1,a2)]['beta']
            sp_act   = act['p2'] - beta_is*act['p1']
            fc       = ou_fc[(a1,a2)]; pcts=fc['pcts']
            fig,ax   = plt.subplots(figsize=(18,6))
            hist     = pair_data[(a1,a2)]['df']['spread_static'].loc['2021':'2023-08-04']
            ax.plot(hist.index, hist.values, color='black', lw=0.8, label='Historical spread')
            ax.fill_between(future_dates,pcts[5], pcts[95],alpha=0.10,color='royalblue',label='90% CI')
            ax.fill_between(future_dates,pcts[25],pcts[75],alpha=0.22,color='royalblue',label='50% CI')
            ax.plot(future_dates,pcts[50],color='royalblue',lw=1.5,ls='--',label='Median forecast')
            ax.plot(sp_act.loc['2023-08-07':].index,
                    sp_act.loc['2023-08-07':].values,
                    color='crimson',lw=1.8,label='Actual 2023–2026',zorder=7)
            ax.axhline(fc['theta'],color='green',ls=':',lw=1.2,label=f"θ={fc['theta']:.2f}")
            if fc['med_date']:
                ax.axvline(fc['med_date'],color='orange',ls='--',lw=1.5,
                           label=f"Predicted conv. {fc['med_date'].strftime('%b %Y')}")
            ax.set_title(f'{a1} / {a2} — Forecast vs Actual (2023–2026)',fontweight='bold')
            ax.legend(fontsize=9); plt.tight_layout(); plt.show()
        except Exception as e:
            print(f'  {a1}/{a2}: yfinance error — {e}')
except ImportError:
    print('yfinance not installed. Run: pip install yfinance')


## 14. Summary of Design Decisions & Key Results

### Design Decisions
| Decision | What Was Done | Why |
|---|---|---|
| Dropped Gasoline | 24.5% missing | Too much imputation distortion |
| ADF + KPSS both | Two-sided stationarity test | Cross-validates I(1); catches ambiguous series |
| 4 economically valid pairs | WTI/Brent, Gas Oil/Diesel, Soybeans/Meal, Corn/Wheat | Statistical + economic + unit-compatible filter |
| Rejected Sugar/Coffee | β=7.1 is price-ratio artifact | Spread=−895 has no economic anchor |
| Rejected Aluminium/Nickel | No supply-chain link | Spurious cointegration from macro co-movement |
| Rejected Gas Oil/HRW Wheat | Energy vs grain, no mechanism | Same: spurious |
| Static + Rolling + VECM | All three compared | Rank by OOS Sharpe; understand what each adds |
| Half-life-based rolling window | 2×HL per pair (20–172 days) | Ties estimation window to each pair's own mean-reversion speed; fixed 60-day window was inconsistent with HL |
| Stateful signal loop | State machine, not vectorised | Prevents re-entering same position every bar |
| Stop loss at \|Z\| = 4 | Immediate position close | Caps tail risk during structural breaks |
| P&L normalised by σ | Divide by spread std | Enables cross-pair Sharpe comparison |
| Per-trade win rate | Track round-trips not days | Per-day ≈ 50% for any mean-reverting strategy |
| Walk-forward CV | Expanding window 5 folds | No look-ahead; time ordering preserved |
| OOS: train 2000–2019 | Frozen params, test 2020–2023 | COVID + Russia-Ukraine = genuine stress test |
| Granger on first-differences | Stationary input | Causality test invalid on I(1) levels |
| Nuanced Granger labels | Detect dominant vs weak direction | 'Bidirectional' misleads when one direction is marginal |
| OU on full history | Avoids regime-specific sub-period | Sub-period fitting gave 2-day half-life → trivial forecasts |
| OU train/test split | Last 6 months as holdout | Concrete accuracy metrics; reveals regime breaks |

### Key Results (from actual outputs)
| Pair | Half-life | IS Sharpe | OOS Sharpe | OOS Trades | Finding |
|---|---|---|---|---|---|
| Gas Oil / ULS Diesel | 5.8 days | 1.664 | **−0.801** | 47 | Best IS pair — fails OOS due to Russia-Ukraine regime break |
| WTI / Brent | 21.9 days | 0.312 | −0.436 | 2 | Sanctions compressed spread — 0 signals in 4/5 CV folds |
| **Soybeans / Meal** | 85.8 days | 0.156 | **+0.913** | 3 | Best OOS — crush economics hold across all regimes |
| Corn / Wheat | 83.0 days | 0.281 | +0.112 | 3 | Positive OOS — feed-grain substitution is durable |

### Main Conclusion
**Agricultural pairs beat energy pairs out-of-sample.** Energy spreads are sensitive to geopolitical shocks (sanctions, pipeline rerouting). Agricultural spreads are enforced by physical supply-chain biology and substitution economics — these mechanisms function regardless of geopolitics. The stop loss at |Z|=4 was critical for limiting energy losses during the 2022 disruption.
